In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal

# WERSJA JEDNA KOMÓRKA: całe zadanie jest poniżej.
# Współczynniki filtru: b = [b0, b1, b2], a = [1, a1, a2].
b = [0.5, 0.9, 0.3]
a = [1, 0.25, 0.9]

# Własna funkcja filtru z równania różnicowego.
def filtr(x, b, a):
    y = np.zeros(len(x))

    for n in range(len(x)):
        # Dla początku sygnału brakujące próbki z przeszłości zastępujemy zerami.
        x0 = x[n]
        x1 = x[n - 1] if n >= 1 else 0
        x2 = x[n - 2] if n >= 2 else 0
        y1 = y[n - 1] if n >= 1 else 0
        y2 = y[n - 2] if n >= 2 else 0

        y[n] = b[0] * x0 + b[1] * x1 + b[2] * x2 - a[1] * y1 - a[2] * y2

    return y

# a) Odpowiedź impulsowa i b) porównanie z funkcją filter/lfilter.
N = 100
impuls = np.zeros(N)
impuls[0] = 1
h_moja = filtr(impuls, b, a)
h_filter = signal.lfilter(b, a, impuls)

print('Czy odpowiedź impulsowa zgadza się z lfilter?', np.allclose(h_moja, h_filter))
print('Największa różnica:', np.max(np.abs(h_moja - h_filter)))

# c) Charakterystyka częstotliwościowa filtru.
w, H = signal.freqz(b, a, 1024)
f = w / np.pi
H_db = 20 * np.log10(np.abs(H) + 1e-12)
H_faza = np.unwrap(np.angle(H)) * 180 / np.pi

# d) Zera, bieguny i stabilność.
zera, bieguny, wzmocnienie = signal.tf2zpk(b, a)
stabilny = np.all(np.abs(bieguny) < 1)

print('Zera:', zera)
print('Bieguny:', bieguny)
print('Moduły biegunów:', np.abs(bieguny))
print('Filtr stabilny:', stabilny)

# e) Szum gaussowski oraz widma amplitudowe.
np.random.seed(42)
x = np.random.normal(0, 1, 256)
y = filtr(x, b, a)

freq = 2 * np.fft.rfftfreq(len(x))
X_db = 20 * np.log10(np.abs(np.fft.rfft(x)) / len(x) + 1e-12)
Y_db = 20 * np.log10(np.abs(np.fft.rfft(y)) / len(y) + 1e-12)
_, H_fft = signal.freqz(b, a, freq * np.pi)
H_fft_db = 20 * np.log10(np.abs(H_fft) + 1e-12)

# Wszystkie wykresy w jednym oknie graficznym.
fig, ax = plt.subplots(3, 2, figsize=(15, 12), constrained_layout=True)

n = np.arange(N)
ax[0, 0].stem(n, h_moja, basefmt=' ', linefmt='C0-', markerfmt='C0o', label='moja funkcja')
ax[0, 0].plot(n, h_filter, 'C1--', linewidth=2, label='lfilter')
ax[0, 0].set_title('a-b) Odpowiedź impulsowa')
ax[0, 0].set_xlabel('n')
ax[0, 0].set_ylabel('h[n]')
ax[0, 0].legend()

ax[0, 1].plot(f, H_db, linewidth=2)
ax[0, 1].set_title('c) Charakterystyka amplitudowa')
ax[0, 1].set_xlabel('częstotliwość [×π rad/próbkę]')
ax[0, 1].set_ylabel('wzmocnienie [dB]')

ax[1, 0].plot(f, H_faza, color='C3', linewidth=2)
ax[1, 0].set_title('c) Charakterystyka fazowa')
ax[1, 0].set_xlabel('częstotliwość [×π rad/próbkę]')
ax[1, 0].set_ylabel('faza [stopnie]')

kat = np.linspace(0, 2 * np.pi, 400)
ax[1, 1].plot(np.cos(kat), np.sin(kat), 'k--', linewidth=1)
ax[1, 1].scatter(zera.real, zera.imag, marker='o', s=90, facecolors='none', edgecolors='C0', linewidths=2)
ax[1, 1].scatter(bieguny.real, bieguny.imag, marker='x', s=90, color='C3', linewidths=2)

# Podpisy przy punktach zastępują legendę i od razu pokazują, co jest czym.
for i, zero in enumerate(zera, start=1):
    ax[1, 1].text(zero.real + 0.04, zero.imag + 0.04, f'zero {i}', color='C0')

for i, biegun in enumerate(bieguny, start=1):
    ax[1, 1].text(biegun.real + 0.04, biegun.imag + 0.04, f'biegun {i}', color='C3')

ax[1, 1].text(0.45, 0.88, 'okrąg jednostkowy', color='0.25')
ax[1, 1].axhline(0, color='0.5', linewidth=1)
ax[1, 1].axvline(0, color='0.5', linewidth=1)
ax[1, 1].set_aspect('equal', adjustable='box')
ax[1, 1].set_xlim(-1.45, 1.2)
ax[1, 1].set_ylim(-1.2, 1.2)
ax[1, 1].set_title('d) Zera i bieguny')
ax[1, 1].set_xlabel('Re')
ax[1, 1].set_ylabel('Im')

ax[2, 0].plot(freq, X_db, label='x[n] - szum')
ax[2, 0].plot(freq, Y_db, label='y[n] - po filtracji')
ax[2, 0].set_title('e) Widma amplitudowe x[n] i y[n]')
ax[2, 0].set_xlabel('częstotliwość [×π rad/próbkę]')
ax[2, 0].set_ylabel('amplituda [dB]')
ax[2, 0].legend()

# Charakterystykę filtru przesuwamy do poziomu widma szumu, żeby łatwiej porównać kształt.
ax[2, 1].plot(freq, Y_db, label='widmo y[n]')
ax[2, 1].plot(freq, np.median(X_db) + H_fft_db, 'C1--', linewidth=2, label='charakterystyka filtru')
ax[2, 1].set_title('e) Widmo y[n] a charakterystyka filtru')
ax[2, 1].set_xlabel('częstotliwość [×π rad/próbkę]')
ax[2, 1].set_ylabel('amplituda [dB]')
ax[2, 1].legend()

for wykres in ax.flat:
    wykres.grid(True, linestyle='--', alpha=0.4)

fig.suptitle('Analiza filtru cyfrowego', fontsize=16)
plt.show()
